<a href="https://colab.research.google.com/github/HrutikGujarkar/Generative-AI-chatbot-capable-of-generating-emails-reports-and-technical-explanations-/blob/main/Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U transformers accelerate gradio


In [2]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading language model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("LLM loaded successfully!")
print("Device:", model.device)


Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!
Device: cpu


In [3]:

SYSTEM_PROMPTS = {
    "Email Generator": """
You are a professional email writing assistant.

Create emails based on the user's requirements.
Follow these rules:
- Use a suitable subject line.
- Maintain professional email structure.
- Use an appropriate greeting and closing.
- Do not invent names, dates, achievements, or facts.
- Use placeholders when important details are missing.
- Return only the email and a short optional note if necessary.
""",

    "Report Generator": """
You are an expert report writing assistant.

Create clear, structured and informative reports.
Use relevant sections such as:
1. Title
2. Introduction
3. Objectives
4. Main Findings
5. Analysis or Discussion
6. Conclusion
7. Recommendations, when appropriate

Adapt the structure to the user's request.
Do not invent statistics, research findings, or sources.
Clearly identify assumptions and missing information.
""",

    "Technical Explanation": """
You are an experienced technical educator and software engineer.

Explain technical topics accurately and clearly.
Adapt explanations to the user's experience level.
When relevant, include:
1. Definition
2. How it works
3. Step-by-step explanation
4. A practical example
5. Advantages and limitations
6. Applications
7. A short summary

For programming topics, provide correct code examples
when requested and explain the important parts.
Do not fabricate APIs, benchmark results, or references.
""",

    "General Assistant": """
You are a helpful, accurate and professional AI assistant.
Understand the user's intent and answer directly.
Break complex topics into understandable steps.
Use examples when they improve the answer.
Acknowledge uncertainty instead of inventing facts.
"""
}

print("Prompt templates created successfully!")


Prompt templates created successfully!


In [4]:

def generate_response(
    message,
    history=None,
    task="General Assistant",
    tone="Professional",
    audience="Intermediate",
    temperature=0.7,
    max_new_tokens=600
):
    if not message or not message.strip():
        return "Please enter a message or describe what you want to generate."

    history = history or []

    system_prompt = SYSTEM_PROMPTS.get(
        task, SYSTEM_PROMPTS["General Assistant"]
    )

    # Prompt engineering: role + task + tone + audience + constraints
    system_prompt += f"""

Additional instructions:
- Desired tone: {tone}
- Target audience or expertise level: {audience}
- Follow the user's requested format and length.
- Use headings and lists where appropriate.
- Prioritize clarity, relevance, and factual accuracy.
- If the request lacks details, use placeholders or state assumptions.
"""

    messages = [{"role": "system", "content": system_prompt}]

    # Retain a limited number of recent turns for context.
    # Gradio message history contains role/content dictionaries.
    for item in history[-6:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in ("user", "assistant") and isinstance(content, str):
                messages.append({
                    "role": role,
                    "content": content
                })
        elif isinstance(item, (tuple, list)) and len(item) == 2:
            # Compatibility with older Gradio history formats
            user_text, assistant_text = item
            if user_text:
                messages.append({
                    "role": "user", "content": str(user_text)
                })
            if assistant_text:
                messages.append({
                    "role": "assistant", "content": str(assistant_text)
                })

    messages.append({"role": "user", "content": message})

    # Apply the chat template expected by the instruction-tuned model.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=float(temperature) > 0,
            temperature=max(float(temperature), 0.01),
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only newly generated tokens, not the input prompt.
    new_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return response or "The model did not generate a response. Please try again."


# Test the model before creating the interface
test_response = generate_response(
    message="Write a formal email requesting an internship opportunity.",
    task="Email Generator",
    tone="Formal",
    audience="Professional"
)

print(test_response)


Subject: Inquiry Regarding Internship Opportunity

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Full Name] and I am currently pursuing my [Degree Level] in [Field of Study] at [University Name]. I have been following your company closely due to its innovative approach to [Industry/Field], which aligns perfectly with my academic interests and career aspirations.

Enclosed with this email is a copy of my resume, detailing my educational background, extracurricular activities, and relevant skills. Over the past year, I have taken various courses that have significantly contributed to my understanding of [Specific Area Related to Your Field], including [Course Name 1], [Course Name 2], etc. Additionally, I have interned at [Previous Company Name], where I gained valuable experience in [Job Title’s Responsibilities].

I am particularly interested in exploring opportunities within your [Department/Area] as it resonates most with my academic focus and persona

In [5]:

import gradio as gr

def chatbot(
    message,
    history,
    task,
    tone,
    audience,
    temperature,
    max_new_tokens
):
    return generate_response(
        message=message,
        history=history,
        task=task,
        tone=tone,
        audience=audience,
        temperature=temperature,
        max_new_tokens=max_new_tokens
    )


with gr.Blocks(title="GenAI Writing Assistant") as demo:

    gr.Markdown("""
    # GenAI Writing Assistant
    ### Generate emails, reports and technical explanations using an LLM

    Choose a task, describe what you need, and let the AI help.
    """)

    with gr.Row():
        task = gr.Dropdown(
            choices=[
                "Email Generator",
                "Report Generator",
                "Technical Explanation",
                "General Assistant"
            ],
            value="Email Generator",
            label="Choose Task"
        )

        tone = gr.Dropdown(
            choices=[
                "Professional",
                "Formal",
                "Friendly",
                "Concise",
                "Detailed",
                "Academic"
            ],
            value="Professional",
            label="Writing Tone"
        )

        audience = gr.Dropdown(
            choices=[
                "Beginner",
                "Intermediate",
                "Expert",
                "Professional"
            ],
            value="Intermediate",
            label="Target Audience"
        )

    temperature = gr.Slider(
        minimum=0.1,
        maximum=1.0,
        value=0.7,
        step=0.1,
        label="Creativity (Temperature)"
    )

    max_new_tokens = gr.Slider(
        minimum=100,
        maximum=1200,
        value=600,
        step=100,
        label="Maximum Response Length (Tokens)"
    )

    gr.ChatInterface(
        fn=chatbot,
        type="messages",
        additional_inputs=[
            task,
            tone,
            audience,
            temperature,
            max_new_tokens
        ],
        examples=[
            "Write a formal email asking for an internship.",
            "Create a report on the applications of artificial intelligence.",
            "Explain how a convolutional neural network works.",
            "Explain REST APIs with a Python example."
        ],
        save_history=True
    )

demo.launch(share=True, debug=True)



TypeError: ChatInterface.__init__() got an unexpected keyword argument 'type'